# CP4 · Notebook 04 — Entrenar PilotNet adaptado

**Arquitectura**: 4 conv + 2 FC, ~250k params, salida = steering escalar.

**Tiempo**: 5–10 min en CPU (3 épocas sobre 3500 + augmentation).

In [ ]:
import time, json, numpy as np
from pathlib import Path
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
OUT = Path('../outputs'); OUT.mkdir(exist_ok=True)
data = np.load('../datasets/cp4-highway-bc.npz')

## 1. Re-crear los DataLoaders (mismo código que 03)

In [ ]:
class HighwayBCDataset(Dataset):
    def __init__(self, obs, actions, augment=False):
        self.obs, self.actions, self.augment = obs, actions, augment
    def __len__(self): return len(self.obs)
    def __getitem__(self, idx):
        x = self.obs[idx].astype(np.float32) / 255.0
        y = self.actions[idx]
        if self.augment and np.random.rand() < 0.5:
            x = x[:, ::-1, :].copy(); y = -y
        return torch.from_numpy(x.transpose(2, 0, 1)), torch.tensor(y, dtype=torch.float32)

train_loader = DataLoader(HighwayBCDataset(data['train_obs'], data['train_actions'], augment=True),
                          batch_size=64, shuffle=True, num_workers=0)
val_in_loader = DataLoader(HighwayBCDataset(data['val_in_obs'], data['val_in_actions']),
                           batch_size=64, num_workers=0)
val_ood_loader = DataLoader(HighwayBCDataset(data['val_ood_obs'], data['val_ood_actions']),
                            batch_size=64, num_workers=0)
print('loaders OK')

## 2. Arquitectura — PilotNet adaptado

El **PilotNet original** (NVIDIA 2016) usa input 200×66×3 y 5 conv. Aquí adaptamos al input 84×84×3 con menos compute.

In [ ]:
class PilotNetMini(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 24, 5, stride=2),  nn.ReLU(),     # 84 → 40
            nn.Conv2d(24, 36, 5, stride=2), nn.ReLU(),     # 40 → 18
            nn.Conv2d(36, 48, 3, stride=2), nn.ReLU(),     # 18 → 8
            nn.Conv2d(48, 64, 3),           nn.ReLU(),     # 8 → 6
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 6 * 6, 100), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(100, 1), nn.Tanh(),                  # salida en [-1, 1]
        )
    def forward(self, x): return self.head(self.conv(x)).squeeze(-1)

model = PilotNetMini()
n_params = sum(p.numel() for p in model.parameters())
print(f'PilotNetMini  params={n_params/1e3:.1f}k')

# Sanity check
x = torch.randn(2, 3, 84, 84)
print(f'  test forward: {model(x).shape}  range=[{model(x).min():.2f}, {model(x).max():.2f}]')

## 3. Loop de entrenamiento

Loss: MSE. Optimizer: Adam, lr=1e-3. 3 épocas (recortar a 2 si tu CPU es lenta).

In [ ]:
EPOCHS = 3
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
model = model.to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

history = {'train_mse': [], 'val_in_mse': [], 'val_ood_mse': []}

def evaluate(loader):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            total += criterion(pred, y).item() * len(y)
            n += len(y)
    return total / n

for epoch in range(EPOCHS):
    model.train()
    t0 = time.time()
    epoch_loss, epoch_n = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); optimizer.step()
        epoch_loss += loss.item() * len(y)
        epoch_n += len(y)
    train_mse = epoch_loss / epoch_n
    val_in_mse = evaluate(val_in_loader)
    val_ood_mse = evaluate(val_ood_loader)
    history['train_mse'].append(train_mse)
    history['val_in_mse'].append(val_in_mse)
    history['val_ood_mse'].append(val_ood_mse)
    print(f'  ep{epoch+1}/{EPOCHS}  train={train_mse:.4f}  val_in={val_in_mse:.4f}  val_ood={val_ood_mse:.4f}  ({time.time()-t0:.1f}s)')

torch.save(model.state_dict(), OUT / '04_pilotnet.pt')
print(f'\n✅ Modelo guardado en {OUT / "04_pilotnet.pt"}')

## 4. Curvas de loss

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
epochs = range(1, EPOCHS + 1)
ax.plot(epochs, history['train_mse'], 'o-', label='train', color='#4a6fa5')
ax.plot(epochs, history['val_in_mse'], 's-', label='val in-dist', color='#f4a261')
ax.plot(epochs, history['val_ood_mse'], '^-', label='val OOD', color='#DA4544')
ax.set_xlabel('época'); ax.set_ylabel('MSE'); ax.set_title('CP4 — curvas de entrenamiento')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / '04_loss_curves.png', dpi=100, bbox_inches='tight'); plt.show()

with open(OUT / '04_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print(f'✅ Curves: {OUT / "04_loss_curves.png"}')

## 5. Reflexión rápida

Apunta:
- ¿Bajó train loss consistentemente? Si no, el modelo no aprende — revisa lr o épocas.
- ¿Val OOD claramente peor que val in-dist? **Sí esperado**. Si no, sospechosa el split.
- Si val in-dist es **igual** a train loss → tu modelo no está overfit. Si es **mucho menor**, hay leakage. Si es **mucho mayor**, overfit.

Ve a `05_evaluacion.ipynb`.